In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa

import soundfile as sf

import os
import shutil

# 데이터 셔플 시 난수 기반 순서를 생성하기 위해 사용.
import random

SEED = 42
random.seed(SEED)

In [19]:
# signal processing 파라미터 설정
SR = 22050                              # 샘플링 주파수
DURATION = 2.0                          # 오디오 길이 (초단위)
SAMPLES_PER_TRACK = SR * DURATION       # 1개 오디오 샘플당 기대되는 총 샘플 수
N_FFT = 4096                            # 멜 스펙트로그램 생성 시, 사용할 FFT 윈도우 길이
HOP_LENGTH = 128                        # 프레임 간 hop 크기 (값이 작을수록 시간 해상도 증가)
N_MELS = 126                            # mel filter bank 개수 (너무 작으면 정보 깨지고, 너무 크면 이미지 복잡해짐)

# bandwith filter
FMIN = 100                  # 100 Hz
FMAX = 1000                 # 1000 Hz


In [25]:
# 노이즈 추가
def augment_audio(y, sr):
    # noise
    y = y + 0.003 * np.random.randn(len(y))

    # volume
    y = y * np.random.uniform(0.7, 1.3)

    # pitch
    if np.random.rand() > 0.5:
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=np.random.randint(-2, 2))

    # time stretch
    if np.random.rand() > 0.5:
        y = librosa.effects.time_stretch(y, rate=np.random.uniform(0.8, 1.2))

    return y

# 두개 wav 합치기
def mix_audio(hornet_path, bee_path, output_path, sr=22050):
    # 1. load
    y1, _ = librosa.load(hornet_path, sr=sr)
    y2, _ = librosa.load(bee_path, sr=sr)

    # 2. 길이 맞추기 (짧은 쪽 기준)
    min_len = min(len(y1), len(y2))
    y1 = y1[:min_len]
    y2 = y2[:min_len]

    # 3. mix ratio (랜덤)
    alpha = np.random.uniform(0.3, 0.7)

    # 4. mix
    y_mix = alpha * y1 + (1 - alpha) * y2

    # 5. normalize (클리핑 방지)
    y_mix = y_mix / np.max(np.abs(y_mix))

    # 6. 저장
    sf.write(output_path, y_mix, sr)

    #print(f"Saved: {output_path}")

In [26]:

HORNET_FOLDER = r'D:\Works\khivemind\feature_engineering\audio\Hornet'

NORMAL_FOLDER = r'D:\Works\khivemind\feature_engineering\audio\Normal'

MIX_FOLDER = r'D:\Works\khivemind\feature_engineering\audio\MixHornetNormal'

# MIX 폴더 비우기
if os.path.exists(MIX_FOLDER):
    shutil.rmtree(MIX_FOLDER)

os.makedirs(MIX_FOLDER)

hornet_files = [f for f in os.listdir(HORNET_FOLDER) if f.endswith(".wav")]
normal_files = [f for f in os.listdir(NORMAL_FOLDER) if f.endswith(".wav")]


# 임의 갯수의 mix 사운드  만들기
for i in range(100) :
    hornet_file = os.path.join(HORNET_FOLDER, random.choice(hornet_files))
    normal_file = os.path.join(NORMAL_FOLDER, random.choice(normal_files))
    mix_file =  os.path.join(MIX_FOLDER, f"output_Mix_Hornet_Bee_{i}.wav")

    mix_audio(hornet_file,normal_file,mix_file)


mix_files = [f for f in os.listdir(MIX_FOLDER) if f.endswith(".wav")]

print(f"{len(mix_files)} 개의 mix sound 파일 생성 완료")

100 개의 mix sound 파일 생성 완료
